# Covid Prediciton Case using tree models 

Dataset: Covid_data

**Content:**
1. [Install packages](#1)
1. [Load data](#2)
1. [Data preparation & exploration](#3)
1. [Train and validate models](#4)
    1. [XGBoost](#4a)
    1. [Decision Tree CART](#4b)
    1. [Random Forest](#4c)

<a id="1"></a> 
## 1. Install packages

In [ ]:
# Import packages
# 'as' := we abbreviate the package for common use
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
import random  
import xgboost as xgb 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier 
from sklearn import metrics
from sklearn.ensemble import AdaBoostClassifier  
from sklearn.metrics import accuracy_score              

<a id="2"></a> 
## 2. Load data

In [ ]:
# Load data file
file = "..\\data\\Covid Data.csv"
df = pd.read_csv(file, header=0)

# Show the first lines of the data
df.head()  

<a id="3"></a> 
## 3. Data preparation & exploration

In [ ]:
# Creating a function for a descriptive analysis of a column
def descriptive_analysis(column_name):                         
    # Basic statistics
    mean = df[column_name].mean()
    median = df[column_name].median()
    std_dev = df[column_name].std()
    min_value = df[column_name].min()
    max_value = df[column_name].max()

    # Counting unique values
    unique_values = df[column_name].nunique()

    # Frequency distribution of values
    value_counts = df[column_name].value_counts()

    # Generating the description
    description = f"Description of column '{column_name}':\n"
    description += f"Mean: {mean:.2f}\n"
    description += f"Median: {median:.2f}\n"
    description += f"Standard Deviation: {std_dev:.2f}\n"
    description += f"Min: {min_value}\n"
    description += f"Max: {max_value}\n"
    description += f"Unique Values: {unique_values}\n"
    description += f"Value Counts:\n{value_counts}"

    print(description)

    # Plot histogram
    plt.figure(figsize=(8, 6))
    plt.hist(df[column_name].dropna().copy(), bins=100, edgecolor='k', alpha=0.7)
    plt.title(f'Distribution of {column_name}')
    plt.xlabel(column_name)
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()

In [ ]:
# a) Creating new variables of interest 
#     i. Creating a new dummy variable indicating whether someone died (1) or not (0)
df['DIED'] = (df['DATE_DIED'] != '9999-99-99').astype(int)

#     ii.) Creating a new dummy variable indicating whether someone has a positive covid test (1) or not (0)
df['COVID'] = (df['CLASIFFICATION_FINAL'] < 4).astype(int)

In [ ]:
# b) Checking and transforming dummy variables with weird values
#     i. Checking the values of a random dummy variable
VAR = 'ICU'
descriptive_analysis(VAR)


#     ii. Transforming all dummy variables including 97, 98 or 99 value instead of NA-values
 
## Create a list of all dummy variables including 97, 98 or 99 values
columns_to_replace = ['INTUBED','PNEUMONIA','PREGNANT','DIABETES','COPD','ASTHMA','INMSUPR','HIPERTENSION','OTHER_DISEASE','CARDIOVASCULAR','OBESITY','RENAL_CHRONIC','TOBACCO','ICU']

## Covert columns to appropriate data type
df[columns_to_replace] = df[columns_to_replace].astype(pd.Int64Dtype()) 

## Select values to replace
values_to_replace = [97, 98, 99]

## Replace values in selected columns with NA-values
df[columns_to_replace] = df[columns_to_replace].replace(values_to_replace, pd.NA)


In [ ]:
#     ii. Fix the NA-values (hint: be smart about the NA-values)

## All NA-values in the pregnant column for men can be filled in with 0 
df.loc[df['SEX'] == 0, 'PREGNANT'] = df.loc[df['SEX'] == 0, 'PREGNANT'].fillna(0)

## All NA-values in the intubed or ICU column for patients with type 1 (patient went home) can be filled in with 0
df.loc[df['PATIENT_TYPE'] == 1, 'INTUBED'] = df.loc[df['PATIENT_TYPE'] == 1, 'INTUBED'].fillna(0)
df.loc[df['PATIENT_TYPE'] == 1, 'ICU'] = df.loc[df['PATIENT_TYPE'] == 1, 'ICU'].fillna(0)

## Drop all rows still including NA-values
df_cleaned = df.dropna()

In [ ]:
df_dummified = df.copy(deep=True)
df_undummified = df.copy(deep=True)

In [ ]:
#     iii. Transforming all binary variables including a 2 instead of 0
 
## Create a list of all binary variables 
binary_columns = ['USMER','SEX','PATIENT_TYPE','INTUBED','PNEUMONIA','PREGNANT','DIABETES','COPD','ASTHMA','INMSUPR','HIPERTENSION','OTHER_DISEASE','CARDIOVASCULAR','OBESITY','RENAL_CHRONIC','TOBACCO','ICU','DIED','COVID']

## Convert column to appropriate data type if necessary
df[binary_columns] = df[binary_columns].astype(pd.Int64Dtype()) 

# Replace values in selected columns with 0
df[binary_columns] = df[binary_columns].replace(2, 0)

#    iv. Checking the values of the transformed random dummy variable again
descriptive_analysis(VAR)

In [ ]:
# c) Checking the NA values

#    i. Count the number of NA-values
percentages = []
for column_name in binary_columns:
    total_count = df[column_name].count()
    value_counts = df[column_name].value_counts(normalize=True, dropna=False)
    
    percentage_0 = value_counts.get(0, 0) * 100
    percentage_1 = value_counts.get(1, 0) * 100
    percentage_na = value_counts.get(pd.NA, 0) * 100
    
    percentages.append((column_name, percentage_0, percentage_1, percentage_na))

percentages_df = pd.DataFrame(percentages, columns=['Column', 'Percentage_0', 'Percentage_1', 'Percentage_NA'])
print(percentages_df)

In [ ]:
# d) Explore the data
#    i. Create boxplot for two variables
y = 'AGE'
x = 'ICU'
plt.figure(figsize=(7.5,5))
sns.boxplot(x=df_cleaned[x], y=df_cleaned[y])
plt.title("Boxplot of IC admission against age")
plt.ylabel(y)
plt.xlabel(x)
plt.show()

In [ ]:
#    ii. Create barplot for two variables
x = 'ICU'
y = 'INTUBED'
plt.figure(figsize=(7.5,5))
sns.barplot(x=df_cleaned[x], y=df_cleaned[y])
plt.title("Percentage of positve tested patients that are admitted on the IC")
plt.ylabel(y)
plt.xlabel(x)
plt.show()

<a id="4"></a> 
## 4. Train and validate models

In [ ]:
# Splitting the dataset into a train and test set for each step
data_train, data_test  = train_test_split(df_cleaned, test_size=0.3, random_state=123)
#data_train_step1, data_test_step1 = train_test_split(df_step1, test_size=0.3, random_state=123)
#data_train_step2, data_test_step2 = train_test_split(df_step2, test_size=0.3, random_state=123)

__*Exercise:* First, run a cell with a specific step (1, 2, 3 or 4) of the conceptual model. Then, choose which model (A, B, C or D) to use for this step by running the different models on each step and reflecting on their performance measures. Write down your results and argumentation.__

In [ ]:
# Step 1 of the model
X_variables = ['SEX','AGE','PREGNANT','DIABETES','COPD','ASTHMA','INMSUPR','HIPERTENSION','OTHER_DISEASE','CARDIOVASCULAR','OBESITY','RENAL_CHRONIC','TOBACCO']
y_variable = 'COVID'

X_train = data_train.loc[:, X_variables]
y_train = data_train[y_variable].astype('int')
X_test = data_test.loc[:, X_variables]
y_test = data_test[y_variable].astype('int')

In [ ]:
# Step 2 of the model 

X_variables = ['SEX','AGE','PREGNANT','DIABETES','ASTHMA','OBESITY']
y_variable = 'ICU'

X_train = data_train.loc[:, X_variables]
y_train = data_train[y_variable].astype('int')
X_test = data_test.loc[:, X_variables]
y_test = data_test[y_variable].astype('int')

In [ ]:
# Step 3 of the model

X_variables = ['SEX','AGE','PREGNANT','DIABETES','COPD','ASTHMA','INMSUPR','HIPERTENSION','OTHER_DISEASE','CARDIOVASCULAR','OBESITY','RENAL_CHRONIC','TOBACCO','PNEUMONIA']
y_variable = 'INTUBED'

X_train = data_train.loc[:, X_variables]
y_train = data_train[y_variable].astype('int')
X_test = data_test.loc[:, X_variables]
y_test = data_test[y_variable].astype('int')

In [ ]:
# Step 4 of the model
# Here, we again use the dataset created for step 3 

X_variables = ['SEX','AGE','PREGNANT','DIABETES','COPD','ASTHMA','INMSUPR','HIPERTENSION','OTHER_DISEASE','CARDIOVASCULAR','OBESITY','RENAL_CHRONIC','TOBACCO','PNEUMONIA', 'INTUBED']
y_variable = 'DIED'

X_train = data_train.loc[:, X_variables]
y_train = data_train[y_variable].astype('int')
X_test = data_test.loc[:, X_variables]
y_test = data_test[y_variable].astype('int')

<a id="4a"></a> 
### A) XGBoost

In [ ]:
# Set model paramaters
N_trees        = 10                        # Number of trees that are estimated
Max_depth      = 4                           # Maximum depth of each tree (nr of levels)
Learning_rate  = 1                           # The learning rate ('eta')
Min_bucket     = 3                           # Minimum number of items per bucket
Subsample      = 0.5                         # Subsample ratio of the training instance
Silent         = 0                           # Whether to print messages while running boosting

# Estimate the model
XGB = xgb.XGBClassifier(n_estimators=N_trees
                        ,max_depth=Max_depth 
                        ,learning_rate=Learning_rate
                        ,min_child_weight=Min_bucket
                        ,subsample=Subsample
                       )

# Fit the model over the train set
XGB.fit(X_train, y_train)

# Create predictions for the test set
preds_proba = XGB.predict_proba(X_test)
preds = XGB.predict(X_test)   # Cut-off point equals 0.5

pd.concat([pd.DataFrame(preds_proba, columns=["Prob. 0", "Prob. 1"]), pd.DataFrame(preds, columns=["Prediction"])], axis=1).head()

# Evaluate results
# Create confusion matrix
print(pd.crosstab(preds, y_test))

# Create classification report
print(classification_report(y_test, preds))

# Calculate and plot the feature ranking
importances = XGB.feature_importances_ 
std = np.std([XGB.feature_importances_], axis=0)
indices = np.argsort(importances)[::-1]

importances_features = []
print("Feature ranking:")                    # Print the feature ranking
for f in range(X_train.shape[1]):
    print("Feature %d (%s) %f" % (indices[f], X_variables[indices[f]], importances[indices[f]]))
    importances_features.append(X_variables[indices[f]])

plt.figure(figsize=(7.5,5))
plt.figure()                                 # Plot the feature importances 
plt.bar(range(X_train.shape[1]), importances[indices],
       color="r", yerr=std[indices], align="center")
plt.title("Feature importances")
plt.ylabel("Importance in terms of decreasing the weighted impurity")
plt.xlabel("Feature")
plt.xticks(range(X_train.shape[1]), importances_features, rotation = 30)
plt.xlim([-1, X_train.shape[1]])
plt.show()

# Create and plot ROC curve and AUC
fpr, tpr, t = metrics.roc_curve(y_test, preds_proba[:,1])
roc_auc_xgboost = metrics.auc(fpr, tpr)

plt.figure(figsize=(10,6))
plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')      # Plot results
plt.plot(fpr, tpr, lw=2, alpha=0.3, label='Mean ROC XGBoost (AUC = %0.2f)' % (roc_auc_xgboost))
plt.title('ROC curve')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.legend(loc="lower right")
plt.show()

# Plot error with respect to number of estimators (trees)
eval_set = [(X_train, y_train), (X_test, y_test)]
eval_metric = ["auc","error"]
# Estimate the model
# Initialize the XGBoost classifier (note: eval_set is not passed in constructor)
XGB = xgb.XGBClassifier(
    n_estimators=N_trees,
    max_depth=Max_depth,
    learning_rate=Learning_rate,
    min_child_weight=Min_bucket,
    subsample=Subsample,
    eval_metric=eval_metric
)

# Fit the model while passing eval_set and eval_metric to the fit method
trained_xgboost = XGB.fit(X_train, y_train, eval_set=eval_set, verbose=True)
real_test_predict = trained_xgboost.predict(X_test)
predictions = [round(value) for value in real_test_predict]

# Retrieve performance metrics
results = trained_xgboost.evals_result()
epochs = len(results['validation_0']['error'])
x_axis = range(0, epochs)

fig, ax = plt.subplots()
ax.plot(x_axis, results['validation_0']['error'], label='Train')
ax.plot(x_axis, results['validation_1']['error'], label='Test')
ax.legend()
plt.ylabel('Classification Error')
plt.title('XGBoost Classification Error')
plt.show()

<b id="4b"></b> 
### B) Decision tree CART

In [ ]:
# Set model parameters 
Min_num_splits = 5
Min_bucket     = 3
Max_depth      = 5

# Estimate the model
decision_tree = DecisionTreeClassifier(max_depth=Max_depth
                                ,min_samples_split=Min_num_splits
                                ,min_samples_leaf=Min_bucket
                                ,criterion="gini"              
                                ,splitter="best"
                                ,random_state=random.seed()
                                )

# Fit the model over the train set
decision_tree.fit(X_train, y_train) 

# Create predictions for the test set
preds_proba = decision_tree.predict_proba(X_test)
preds = decision_tree.predict(X_test)   # Cut-off point equals 0.5

pd.concat([pd.DataFrame(preds_proba, columns=["Prob. 0", "Prob. 1"]), pd.DataFrame(preds, columns=["Prediction"])], axis=1).head()

# Calculate the optimal cut-off point given these theoretical costs
cost_TP = 0
cost_TN = 0
cost_FP = 50
cost_FN = 100
total_cost = math.inf

for i in np.linspace(0,1,100,endpoint=False):
    y_pred = (preds_proba[:,1]>i).astype('int')
    results = metrics.confusion_matrix(y_pred,y_test)
    TN = results[0][0]
    FN = results[0][1]
    FP = results[1][0]
    TP = results[1][1]
    
    # Calculate cutoff-point
    cost = TN*cost_TN + TP*cost_TP + FP*cost_FP + FN*cost_FN
    total_cost = min(total_cost,cost)
    if(total_cost == cost):
        opt_cutoff = i
        
print('Optimal cut-off:', opt_cutoff)

# Create predictions for the test set according to optimal cutoff point
preds = (preds_proba[:,1] > opt_cutoff).astype('int')

# Evaluate results
# Create confusion matrix
print(pd.crosstab(preds, y_test))

# Create classification report
print(classification_report(y_test, preds))

# Get the feature importances of the tree
importances = decision_tree.feature_importances_ 
std = np.std([decision_tree.feature_importances_], axis=0)
indices = np.argsort(importances)[::-1]

importances_features = []
print("Feature ranking:")                    # Print the feature ranking
for f in range(X_train.shape[1]):
    print("Feature %d (%s) %f" % (indices[f], X_variables[indices[f]], importances[indices[f]]))
    importances_features.append(X_variables[indices[f]])

plt.figure(figsize=(7.5,5))
plt.figure()                                 # Plot the feature importances 
plt.bar(range(X_train.shape[1]), importances[indices],
       color="blue", yerr=std[indices], align="center")
plt.title("Feature importances")
plt.ylabel("Importance in terms of decreasing the weighted impurity")
plt.xlabel("Feature")
plt.xticks(range(X_train.shape[1]), importances_features, rotation = 30)
plt.xlim([-1, X_train.shape[1]])
plt.show()

# Create and plot ROC curve en AUC
fpr, tpr, t = metrics.roc_curve(y_test, preds_proba[:,1])
roc_auc_tree = metrics.auc(fpr, tpr)

plt.figure(figsize=(10,6))
plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')      # Plot results
plt.plot(fpr, tpr, lw=2, alpha=0.3, label='Mean ROC Decision Tree CART (AUC = %0.2f)' % (roc_auc_tree))
plt.title('ROC curve')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.legend(loc="lower right")
plt.show()

<c id="4c"></c> 
### C) Random Forest

In [ ]:
# Set model parameters
N_trees        = 10                        # Number of trees that are estimated
Min_num_splits = 5                           # Minimum number of items to split    
Min_bucket     = 5                           # Minimum number of items per bucket
Max_depth      = 4                           # Maximum depth of each tree (nr of levels)

# Estimate the model
forest = RandomForestClassifier(n_estimators = N_trees
                                ,criterion = "gini"            
                                ,max_depth = Max_depth
                                ,min_samples_split = Min_num_splits
                                ,min_samples_leaf = Min_bucket
                                ,random_state = random.seed()
                                )

# Fit the model over the train set
forest.fit(X_train, y_train)   

# Create predictions for the test set
preds_proba = forest.predict_proba(X_test)
#preds = forest.predict(X_test)                    # Cut-off point equals 0.5
preds = (preds_proba[:,1] > 0.5).astype('int')     # If possible, adjust cut-off value

# Show the first 5 lines of the prediction probabilities and corresponding prediction
pd.concat([pd.DataFrame(preds_proba, columns=["Prob. 0", "Prob. 1"]), pd.DataFrame(preds, columns=["Prediction"])], axis=1).head()

# Evaluate results
# Create confusion matrix
print(pd.crosstab(preds, y_test))

# Create classification report
print(classification_report(y_test, preds))

# Get the feature importances of the forest
importances = forest.feature_importances_ 
std = np.std([forest.feature_importances_ for tree in forest.estimators_], axis=0)
indices = np.argsort(importances)[::-1]

importances_features = []
print("Feature ranking:")                    # Print the feature ranking
for f in range(X_train.shape[1]):
    print("Feature %d (%s) %f" % (indices[f], X_variables[indices[f]], importances[indices[f]]))
    importances_features.append(X_variables[indices[f]])

plt.figure(figsize=(7.5,5))
plt.figure()                                 # Plot the feature importances 
plt.bar(range(X_train.shape[1]), importances[indices],
       color="r", yerr=std[indices], align="center")
plt.title("Feature importances")
plt.ylabel("Importance in terms of decreasing the weighted impurity")
plt.xlabel("Feature")
plt.xticks(range(X_train.shape[1]), importances_features, rotation = 30)
plt.xlim([-1, X_train.shape[1]])
plt.show()
 
# Create and plot ROC curve and AUC
fpr, tpr, t = metrics.roc_curve(y_test, preds_proba[:,1])
roc_auc_forest = metrics.auc(fpr, tpr)

plt.figure(figsize=(10,6))
plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')      # Plot results
plt.plot(fpr, tpr, lw=2, alpha=0.3, label='Mean ROC Random Forest (AUC = %0.2f)' % (roc_auc_forest))
plt.title('ROC curve')
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.legend(loc="lower right")
plt.show()